# ML-09 — Validation and Research Claim Audit

This notebook distinguishes descriptive clustering evidence from causal or supervised-model claims.

## 1. Questions the validation must answer

1. Does the cluster structure remain reasonably coherent for pages from held-out clients?
2. Are any final features label-derived, future-looking, proprietary product outputs, or private identifiers?

It does not ask whether a refresh causes traffic recovery: the snapshot data and this design cannot answer that causal question.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
df_clean = df_raw.loc[
    (df_raw['impressions_90d'] >= 10)
    & (df_raw['content_age_days'] >= 90)
    & (df_raw['avg_position'] > 0)
] .copy()

X = pd.DataFrame({
    'impressions_log': np.log1p(df_clean['impressions_90d']),
    'avg_position': df_clean['avg_position'],
    'ctr': df_clean['ctr'],
    'staleness_log': np.log1p(df_clean['days_since_last_update']),
    'engagement_rate': df_clean['engagement_rate'],
})
print(f'Validation corpus: {len(df_clean):,} pages across {df_clean["client_id"].nunique()} pseudonymized clients')


## 2. Random split versus client holdout

The random split is shown only as a leakage-risk contrast. The primary result is the client-held-out cluster coherence: fit centroids on reference clients, assign held-out observations, then calculate silhouette on the held-out assignments. This is not classification accuracy and it does not prove generalization.

In [ ]:
X_train_rand, X_test_rand = train_test_split(X, test_size=0.20, random_state=42)
random_scaler = StandardScaler()
X_train_rand_scaled = random_scaler.fit_transform(X_train_rand)
X_test_rand_scaled = random_scaler.transform(X_test_rand)
random_model = KMeans(n_clusters=5, random_state=42, n_init=20).fit(X_train_rand_scaled)
random_labels = random_model.predict(X_test_rand_scaled)

rng = np.random.default_rng(42)
holdout_clients = rng.choice(
    df_clean['client_id'].unique(),
    size=max(1, round(df_clean['client_id'].nunique() * 0.20)),
    replace=False,
)
train_mask = ~df_clean['client_id'].isin(holdout_clients)
test_mask = ~train_mask
group_scaler = StandardScaler()
X_train_group_scaled = group_scaler.fit_transform(X.loc[train_mask])
X_test_group_scaled = group_scaler.transform(X.loc[test_mask])
group_model = KMeans(n_clusters=5, random_state=42, n_init=20).fit(X_train_group_scaled)
group_labels = group_model.predict(X_test_group_scaled)

random_sample = np.arange(0, len(X_test_rand_scaled), 10)
group_sample = np.arange(0, len(X_test_group_scaled), 10)
split_comparison = pd.DataFrame({
    'design': ['Random page split (contrast only)', 'Client-held-out split (primary)'],
    'held_out_pages': [len(X_test_rand_scaled), len(X_test_group_scaled)],
    'held_out_clients': ['mixed', df_clean.loc[test_mask, 'client_id'].nunique()],
    'held_out_silhouette': [
        silhouette_score(X_test_rand_scaled[random_sample], random_labels[random_sample]),
        silhouette_score(X_test_group_scaled[group_sample], group_labels[group_sample]),
    ],
})
display(split_comparison.round(3))


## 3. Final feature leakage and decision-time audit

Correlation with a decline proxy is reported as a diagnostic, not as the sole definition of leakage. The stronger safeguards are explicit exclusion of target-derived and product-output fields, use of only observed-window variables, and removal of identifiers from the model matrix.

In [ ]:
decline_proxy = (df_clean['trend_direction'] == 'down').astype(int)
feature_corrs = X.corrwith(decline_proxy).rename('correlation_with_decline_proxy')
display(feature_corrs.to_frame().round(3))

forbidden_fields = {
    'trend_direction', 'trend_pct', 'is_declining_label',
    'health_score', 'priority_score', 'action_type',
    'client_id', 'content_id', 'url', 'keyword_text',
}
assert not (set(X.columns) & forbidden_fields)
assert df_clean['avg_position'].gt(0).all()
print('Passed: feature matrix contains only the five audited decision-time dimensions.')


## 4. Calibrated claim

**Supported:** The held-out client evaluation provides evidence that the resulting cluster structure remains reasonably coherent outside the clients used for the original fit. The archetypes are descriptive decision-support categories.

**Not supported:** The clustering proves that stale content causes ranking loss, that a heuristic score causes traffic recovery, or that recommendations will improve outcomes without a controlled intervention study.

## Skeptic check

- Clustering is descriptive, and K is a modelling choice.
- Cluster labels are human interpretations of arbitrary model IDs.
- Client mix and feature definitions can change the result.
- Observational telemetry cannot prove intervention impact.
- Any action should pass human editorial, SERP-intent, and YMYL review.

## Self-check

- [x] Held-out coherence is distinguished from classification performance.
- [x] Leakage controls are explicit.
- [x] Claims do not exceed the evidence.